# News Data Exploration

This notebook loads and explores financial news datasets for sentiment analysis and NLP tasks.

**Dataset:**
- `All_external.csv` - Financial news articles from Benzinga and other sources with multiple summary types

## 1. Setup and Imports

In [3]:
import pandas as pd
import numpy as np
from pathlib import Path
from collections import Counter
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.max_colwidth', 100)

print("Libraries loaded successfully!")

Libraries loaded successfully!


## 2. Load News Dataset

In [4]:
NEWS_DIR = Path("../data/raw/news")

print("Available news files:")
for f in NEWS_DIR.iterdir():
    if f.is_file() and not f.name.startswith('.'):
        size_mb = f.stat().st_size / (1024 * 1024)
        print(f"  {f.name} ({size_mb:.2f} MB)")

Available news files:
  AAPL_news.csv (0.06 MB)
  All_external.csv (5465.89 MB)
  AMZN_news.csv (0.03 MB)
  MAG7_news.csv (1.01 MB)


### 2.1 Load All_external.csv

In [5]:
mag7_external_path = NEWS_DIR / "MAG7_news.csv"

if mag7_external_path.exists():
    df_all_news = pd.read_csv(mag7_external_path)
    print(f"All_external.csv loaded successfully!")
    print(f"   Shape: {df_all_news.shape}")
    print(f"   Columns: {list(df_all_news.columns)}")
else:
    print(f"File not found: {mag7_external_path}")

All_external.csv loaded successfully!
   Shape: (4439, 11)
   Columns: ['Date', 'Article_title', 'Stock_symbol', 'Url', 'Publisher', 'Author', 'Article', 'Lsa_summary', 'Luhn_summary', 'Textrank_summary', 'Lexrank_summary']


In [6]:
# Display first few rows
print("First 5 rows of All_external.csv:")
df_all_news.head()

First 5 rows of All_external.csv:


,Date,Article_title,Stock_symbol,Url,Publisher,Author,Article,Lsa_summary,Luhn_summary,Textrank_summary,Lexrank_summary
0,2020-06-10 07:33:26 UTC,Tech Stocks And FAANGS Strong Again To Start Day As Market Awaits Fed,AAPL,https://www.benzinga.com/government/20/06/16223418/tech-stocks-and-faangs-strong-again-to-start-...,JJ Kinahan,NaN,NaN,NaN,NaN,NaN,NaN
1,2020-06-10 04:14:08 UTC,10 Biggest Price Target Changes For Wednesday,AAPL,https://www.benzinga.com/analyst-ratings/price-target/20/06/16220539/10-biggest-price-target-cha...,Lisa Levin,NaN,NaN,NaN,NaN,NaN,NaN
2,2020-06-09 20:52:01 UTC,Big Tech Reaches New Record Heights At The Stock Market,AAPL,https://www.benzinga.com/news/20/06/16218615/big-tech-reaches-new-record-heights-at-the-stock-ma...,Neer Varshney,NaN,NaN,NaN,NaN,NaN,NaN
3,2020-06-09 11:14:07 UTC,Why Apple's Stock Is Trading Higher Today,AAPL,https://www.benzinga.com/news/20/06/16215446/why-apples-stock-is-trading-higher-today,Tanzeel Akhtar,NaN,NaN,NaN,NaN,NaN,NaN
4,2020-06-09 09:58:46 UTC,Apple Could Announce In-House Chips For Macs At WWDC: Report,AAPL,https://www.benzinga.com/news/20/06/16214115/apple-could-announce-in-house-chips-for-macs-at-wwd...,Shanthi Rexaline,NaN,NaN,NaN,NaN,NaN,NaN


In [7]:
# Data types
print("Data Types:")
print(df_all_news.dtypes)
print()
print(f"Memory Usage: {df_all_news.memory_usage(deep=True).sum() / 1024**2:.2f} MB")

Data Types:
Date                    str
Article_title           str
Stock_symbol            str
Url                     str
Publisher               str
Author              float64
Article             float64
Lsa_summary         float64
Luhn_summary        float64
Textrank_summary    float64
Lexrank_summary     float64
dtype: object

Memory Usage: 1.33 MB


In [8]:
# Missing values
print("Missing values:")
print(df_all_news.isnull().sum())
print()
df_all_news.info()

Missing values:
Date                   0
Article_title          0
Stock_symbol           1
Url                    0
Publisher              1
Author              4439
Article             4439
Lsa_summary         4439
Luhn_summary        4439
Textrank_summary    4439
Lexrank_summary     4439
dtype: int64

<class 'pandas.DataFrame'>
RangeIndex: 4439 entries, 0 to 4438
Data columns (total 11 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   Date              4439 non-null   str    
 1   Article_title     4439 non-null   str    
 2   Stock_symbol      4438 non-null   str    
 3   Url               4439 non-null   str    
 4   Publisher         4438 non-null   str    
 5   Author            0 non-null      float64
 6   Article           0 non-null      float64
 7   Lsa_summary       0 non-null      float64
 8   Luhn_summary      0 non-null      float64
 9   Textrank_summary  0 non-null      float64
 10  Lexrank_summary   0 non-null  

## 3. Explore All_external.csv Dataset

In [9]:
if 'Date' in df_all_news.columns:
    df_all_news['Date'] = pd.to_datetime(df_all_news['Date'], utc=True)
    print(f"Date Range: {df_all_news['Date'].min()} to {df_all_news['Date'].max()}")

print(f"Total Articles: {len(df_all_news):,}")

if 'Stock_symbol' in df_all_news.columns:
    print(f"Unique Stock Symbols: {df_all_news['Stock_symbol'].nunique()}")

if 'Publisher' in df_all_news.columns:
    print(f"Unique Publishers: {df_all_news['Publisher'].nunique()}")

Date Range: 2009-05-11 14:00:00+00:00 to 2020-06-10 11:08:09+00:00
Total Articles: 4,439
Unique Stock Symbols: 7
Unique Publishers: 180


In [10]:
# Top stock symbols by article count
if 'Stock_symbol' in df_all_news.columns:
    print("Top 20 Stock Symbols by Article Count:")
    top_symbols = df_all_news['Stock_symbol'].value_counts().head(20)
    print(top_symbols.to_string())

Top 20 Stock Symbols by Article Count:
Stock_symbol
NVDA     1857
TSLA     1151
GOOGL    1018
AAPL      268
AMZN      142
NWS         1
XRX         1


In [11]:
# Top publishers
if 'Publisher' in df_all_news.columns:
    print("Top 10 Publishers:")
    top_publishers = df_all_news['Publisher'].value_counts().head(10)
    print(top_publishers.to_string())

Top 10 Publishers:
Publisher
Benzinga Newsdesk    973
Lisa Levin           483
Charles Gross        393
Wayne Duggan         205
Jayson Derrick       166
Neer Varshney        151
Paul Quintaro        138
Shanthi Rexaline     120
JJ Kinahan            97
IAM Newswire          90


In [12]:
# Check article content availability
if 'Article' in df_all_news.columns:
    article_str = df_all_news['Article'].astype(str)
    has_article = (df_all_news['Article'].notna()) & (article_str != 'N/A') & (article_str.str.strip() != '') & (article_str.str.strip().str.lower() != 'nan')
    print(f"Articles with full text: {has_article.sum():,} ({has_article.mean()*100:.1f}%)")

# Check summary availability
for summary_col in ['Lsa_summary', 'Luhn_summary', 'Textrank_summary', 'Lexrank_summary']:
    if summary_col in df_all_news.columns:
        summary_str = df_all_news[summary_col].astype(str)
        has_summary = (df_all_news[summary_col].notna()) & (summary_str != 'N/A') & (summary_str.str.strip() != '') & (summary_str.str.strip().str.lower() != 'nan')
        print(f"{summary_col}: {has_summary.sum():,} available ({has_summary.mean()*100:.1f}%)")

Articles with full text: 0 (0.0%)
Lsa_summary: 0 available (0.0%)
Luhn_summary: 0 available (0.0%)
Textrank_summary: 0 available (0.0%)
Lexrank_summary: 0 available (0.0%)


## 4. Filter News for Target Stocks

In [13]:
TARGET_TICKERS = ["AAPL", "MSFT", "GOOGL", "AMZN", "TSLA", "NVDA", "META"]

# Filter All_external for our target stocks
if 'Stock_symbol' in df_all_news.columns:
    df_target = df_all_news[df_all_news['Stock_symbol'].isin(TARGET_TICKERS)].copy()
    print(f"Articles for target tickers: {len(df_target):,} out of {len(df_all_news):,} total")
    print()
    print("Articles per target ticker:")
    print(df_target['Stock_symbol'].value_counts().to_string())

Articles for target tickers: 4,436 out of 4,439 total

Articles per target ticker:
Stock_symbol
NVDA     1857
TSLA     1151
GOOGL    1018
AAPL      268
AMZN      142


In [14]:
# Date distribution for target stocks
if 'Date' in df_target.columns:
    print(f"Date Range for target tickers: {df_target['Date'].min()} to {df_target['Date'].max()}")
    print()
    print("Articles per year:")
    print(df_target['Date'].dt.year.value_counts().sort_index().to_string())

Date Range for target tickers: 2011-03-03 00:00:00+00:00 to 2020-06-10 11:08:09+00:00

Articles per year:
Date
2011     135
2012     107
2013      82
2014      74
2015     114
2016     237
2017     405
2018     443
2019    1232
2020    1607


In [ ]:
filtered_external_path = NEWS_DIR / "Filtered_external.csv"
df_target.to_csv(filtered_external_path)

## 5. Sample Article Content

In [16]:
# Display a sample article from All_external.csv
print("SAMPLE ARTICLE FROM All_external.csv")
print("=" * 60)
sample = df_all_news.iloc[0]
for col in df_all_news.columns:
    val = str(sample[col])[:200]
    print(f"{col}: {val}")
    print()

SAMPLE ARTICLE FROM All_external.csv
Date: 2020-06-10 07:33:26+00:00

Article_title: Tech Stocks And FAANGS Strong Again To Start Day As Market Awaits Fed

Stock_symbol: AAPL

Url: https://www.benzinga.com/government/20/06/16223418/tech-stocks-and-faangs-strong-again-to-start-day-as-market-awaits-fed

Publisher: JJ Kinahan

Author: nan

Article: nan

Lsa_summary: nan

Luhn_summary: nan

Textrank_summary: nan

Lexrank_summary: nan



In [17]:
# Display a sample article from target tickers that has content
if len(df_target) > 0:
    # Try to find an article with actual content
    article_str = df_target['Article'].astype(str)
    has_content = df_target[
        (df_target['Article'].notna()) & 
        (article_str != 'N/A') & 
        (article_str.str.len() > 50)
    ]
    if len(has_content) > 0:
        sample_target = has_content.iloc[0]
        print(f"SAMPLE ARTICLE FOR {sample_target['Stock_symbol']}")
        print("=" * 60)
        for col in df_target.columns:
            val = str(sample_target[col])[:300]
            print(f"{col}: {val}")
            print()
    else:
        print("No articles with substantial content found for target tickers.")
        sample_target = df_target.iloc[0]
        print(f"\nSAMPLE (title only) FOR {sample_target['Stock_symbol']}:")
        print(f"Title: {sample_target.get('Article_title', 'N/A')}")
        print(f"Date: {sample_target.get('Date', 'N/A')}")

No articles with substantial content found for target tickers.

SAMPLE (title only) FOR AAPL:
Title: Tech Stocks And FAANGS Strong Again To Start Day As Market Awaits Fed
Date: 2020-06-10 07:33:26+00:00


## 6. Summary

Key findings from news data exploration:
- Total number of articles in the dataset
- Date range covered
- Stock symbol coverage for our target tickers (AAPL, MSFT, GOOGL, AMZN, TSLA, NVDA, META)
- Data quality (missing values, empty fields)
- Summary text availability (LSA, Luhn, TextRank, LexRank)